In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import os
os.chdir('/content/drive/MyDrive/credit-risk-assessment-system')

import pandas as pd
import numpy as np
import joblib
import json

X_train = pd.read_csv('data/processed/X_train_fe.csv')
X_test = pd.read_csv('data/processed/X_test_fe.csv')
y_test = pd.read_csv('data/processed/y_test.csv').squeeze()

leftover_cols = [c for c in ['SK_ID_CURR', 'AMT_ANNUITY_RAW', 'AMT_CREDIT_RAW', 'AMT_INCOME_TOTAL_RAW',
                               'CODE_GENDER_M', 'CODE_GENDER_XNA']
                  if c in X_train.columns]
X_train_fair = X_train.drop(columns=leftover_cols)
X_test_fair = X_test.drop(columns=leftover_cols)

catboost_fair = joblib.load('models/catboost_final_fair.pkl')

with open('models/decision_threshold_final.json') as f:
    decision_config = json.load(f)

print(X_train_fair.shape, X_test_fair.shape)
print(decision_config)

(246008, 189) (61503, 189)
{'optimal_threshold': 0.7198832270177709, 'precision_at_threshold': 0.29607250755287007, 'recall_at_threshold': 0.29607250755287007, 'features_excluded_for_fairness': ['CODE_GENDER_M', 'CODE_GENDER_XNA'], 'auc': 0.760375402945312}


In [25]:
y_pred_proba = catboost_fair.predict_proba(X_test_fair)[:, 1]

def assign_risk_band(prob):
    if prob < 0.10:
        return 'Low'
    elif prob < 0.30:
        return 'Medium'
    else:
        return 'High'

risk_bands = pd.Series(y_pred_proba).apply(assign_risk_band)
print(risk_bands.value_counts())
print(risk_bands.value_counts(normalize=True) * 100)

High      39856
Medium    20010
Low        1637
Name: count, dtype: int64
High      64.803343
Medium    32.534998
Low        2.661659
Name: proportion, dtype: float64


In [26]:
validation_df = pd.DataFrame({
    'risk_band': risk_bands.values,
    'actual_default': y_test.values
})

band_validation = validation_df.groupby('risk_band')['actual_default'].agg(['mean', 'count'])
band_validation = band_validation.reindex(['Low', 'Medium', 'High'])
print(band_validation)

               mean  count
risk_band                 
Low        0.007941   1637
Medium     0.022839  20010
High       0.112781  39856


In [27]:
low_cutoff = np.percentile(y_pred_proba, 50)   # bottom 50% = Low
high_cutoff = np.percentile(y_pred_proba, 85)  # top 15% = High

print(f"Low cutoff: {low_cutoff:.4f}, High cutoff: {high_cutoff:.4f}")

def assign_risk_band_quantile(prob):
    if prob < low_cutoff:
        return 'Low'
    elif prob < high_cutoff:
        return 'Medium'
    else:
        return 'High'

risk_bands_v2 = pd.Series(y_pred_proba).apply(assign_risk_band_quantile)
print(risk_bands_v2.value_counts())

validation_df_v2 = pd.DataFrame({'risk_band': risk_bands_v2.values, 'actual_default': y_test.values})
band_validation_v2 = validation_df_v2.groupby('risk_band')['actual_default'].agg(['mean', 'count']).reindex(['Low', 'Medium', 'High'])
print(band_validation_v2)

Low cutoff: 0.3825, High cutoff: 0.6402
Low       30751
Medium    21526
High       9226
Name: count, dtype: int64
               mean  count
risk_band                 
Low        0.028097  30751
Medium     0.088683  21526
High       0.237589   9226


In [28]:
def probability_to_score(prob, min_score=300, max_score=850):
    # Invert: high probability of default = low score
    score = max_score - (prob * (max_score - min_score))
    return int(np.clip(score, min_score, max_score))

credit_scores = pd.Series(y_pred_proba).apply(probability_to_score)

print(credit_scores.describe())

count    61503.000000
mean       625.782937
std        109.273294
min        316.000000
25%        548.000000
50%        639.000000
75%        714.000000
max        846.000000
dtype: float64


In [29]:
scoring_results = pd.DataFrame({
    'default_probability': y_pred_proba,
    'risk_band': risk_bands_v2.values,
    'credit_score': credit_scores.values,
    'actual_default': y_test.values
})

print(scoring_results.head(10))

scoring_results.to_csv('reports/risk_scoring_results.csv', index=False)

scoring_config = {
    'low_cutoff': float(low_cutoff),
    'high_cutoff': float(high_cutoff),
    'score_range': [300, 850]
}
with open('models/scoring_config.json', 'w') as f:
    json.dump(scoring_config, f, indent=2)

print("Saved scoring results and config")

   default_probability risk_band  credit_score  actual_default
0             0.245808       Low           714               0
1             0.473865    Medium           589               0
2             0.729333      High           448               0
3             0.510585    Medium           569               0
4             0.515677    Medium           566               0
5             0.645497      High           494               0
6             0.162238       Low           760               0
7             0.102538       Low           793               0
8             0.835573      High           390               0
9             0.481364    Medium           585               1
Saved scoring results and config
